In [7]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


RESET

In [13]:
got.mechanical_joint_control(0, 90, 90, 1000)    
got.mechanical_clamp_close()
got.mecanum_stop()

STRAIGHT

In [ ]:
got.mechanical_joint_control(0, 0, 0, 10)    

CLOSE


In [17]:
got.mechanical_clamp_close()

OPEN

In [16]:
got.mechanical_clamp_release()

MANUAL CONTROL

FWD

In [69]:
try: 
    while True:
        got.mecanum_translate_speed(angle=0,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


LEFT

In [3]:
try: 
    while True:
        got.mecanum_translate_speed(angle=-90,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")

NameError: name 'got' is not defined

RIGHT

In [ ]:
try: 
    while True:
        got.mecanum_translate_speed(angle=90,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")

BACK

In [20]:
try: 
    while True:
        got.mecanum_translate_speed(angle=180,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


Full reset arm

In [8]:
import time

got.mechanical_clamp_release() # Open the gripper
time.sleep(2)
got.mechanical_clamp_close() # Close the gripper

got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-70, duration=1000)

SPIN RIGHT

In [66]:
try: 
    while True:
        got.mecanum_turn_speed(turn=3,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


SPIN LEFT

In [ ]:
try: 
    while True:
        got.mecanum_turn_speed(turn=2,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")

Camera


In [ ]:
import cv2
import numpy as np
import time

try:
    from ugot import ugot
except:
    ugot = None


# ---------------------------
# INIT ROBOT
# ---------------------------
def init_robot(ip="192.168.88.1"):
    if ugot is None:
        print("❌ UGOT SDK not installed")
        return None

    try:
        robot = ugot.UGOT()
        robot.initialize(ip)

        try:
            robot.open_camera()
        except:
            pass

        print("✅ Robot connected + camera opened")
        return robot

    except Exception as e:
        print("❌ Failed:", e)
        return None


# ---------------------------
# CAMERA STREAM LOOP
# ---------------------------
def show_robot_camera(robot):
    print("📡 Starting camera stream... Press Q to exit")

    while True:
        try:
            frame = None

            # different SDK versions use different methods
            if hasattr(robot, "get_camera_frame"):
                frame = robot.get_camera_frame()

            elif hasattr(robot, "read_camera_data"):
                raw = robot.read_camera_data()
                if raw is not None:
                    nparr = np.frombuffer(raw, np.uint8)
                    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if frame is None:
                continue

            cv2.imshow("UGOT Robot Camera", frame)

        except Exception as e:
            print("Camera error:", e)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()


# ---------------------------
# RUN
# ---------------------------
if __name__ == "__main__":
    robot = init_robot("192.168.88.1")

    if robot is not None:
        show_robot_camera(robot)

Pose

In [ ]:
import cv2

import numpy as np

import time

from ultralytics import YOLO



try:

    from ugot import ugot

except:

    ugot = None





# ---------------------------

# KEYPOINTS

# ---------------------------

COCO_KEYPOINTS = [

    "nose", "left_eye", "right_eye", "left_ear", "right_ear",

    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",

    "left_wrist", "right_wrist", "left_hip", "right_hip",

    "left_knee", "right_knee", "left_ankle", "right_ankle"

]





# ---------------------------

# INIT ROBOT (SAFE)

# ---------------------------

def init_robot(ip="192.168.88.1"):

    if ugot is None:

        print("❌ UGOT not installed")

        return None



    try:

        robot = ugot.UGOT()

        robot.initialize(ip)



        try:

            robot.load_models([])

        except:

            pass



        try:

            robot.open_camera()

        except:

            pass



        print("✅ Robot object created (camera may still depend on SDK/service)")

        return robot



    except Exception as e:

        print("❌ Robot init failed:", e)

        return None





# ---------------------------

# ROBOT CAMERA (ROBUST + SAFE)

# ---------------------------

def get_robot_frame(robot):

    if robot is None:

        return None



    frame = None



    try:

        if hasattr(robot, "get_camera_frame"):

            frame = robot.get_camera_frame()



        elif hasattr(robot, "read_camera_data"):

            raw = robot.read_camera_data()

            if raw is not None:

                nparr = np.frombuffer(raw, np.uint8)

                frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)



        elif hasattr(robot, "camera"):

            raw = robot.camera.read()

            if raw is not None:

                frame = raw



        if frame is None:

            return None



        frame = cv2.resize(frame, (660, 480))

        return frame



    except:

        return None





# ---------------------------

# SELECT MAIN PERSON

# ---------------------------

def select_main_person(results):

    if not results or results[0].keypoints is None:

        return None



    data = results[0].keypoints.data

    if data is None or len(data) == 0:

        return None



    best_i = 0

    best_score = -1



    for i in range(len(data)):

        person = data[i].detach().cpu().numpy()



        xy = person[:, :2]

        conf = person[:, 2]



        valid = xy[conf > 0.3]

        if len(valid) < 5:

            continue



        score = np.linalg.norm(valid.max(axis=0) - valid.min(axis=0))



        if score > best_score:

            best_score = score

            best_i = i



    return data[best_i].detach().cpu().numpy()





# ---------------------------

# CLASSIFIER

# ---------------------------

def classify_pose(kp, min_conf=0.3):



    def get(name):

        if name not in kp:

            return None

        x, y, c = kp[name]

        if c < min_conf:

            return None

        return np.array([x, y], dtype=np.float32)



    ls = get("left_shoulder")

    rs = get("right_shoulder")

    lw = get("left_wrist")

    rw = get("right_wrist")



    if ls is None or rs is None or lw is None or rw is None:

        return "NONE"



    torso = np.linalg.norm(ls - rs)

    if torso < 1e-6:

        return "NONE"



    up = 0.25 * torso

    down = 0.25 * torso



    left_up = lw[1] < ls[1] - up

    right_up = rw[1] < rs[1] - up



    left_down = lw[1] > ls[1] + down

    right_down = rw[1] > rs[1] + down



    left_mid = not left_up and not left_down

    right_mid = not right_up and not right_down



    dist = abs(lw[0] - rw[0])



    if left_mid and right_mid and dist < 0.55 * torso:

        return "EXIT"



    if left_mid and right_mid and dist > 1.45 * torso:

        return "PICKUP"



    if left_up and right_up:

        return "FORWARD"

    elif left_down and right_down:

        return "BACKWARD"

    elif left_up:

        return "LEFT"

    elif right_up:

        return "RIGHT"



    return "NONE"





# ---------------------------

# DOT SKELETON ONLY

# ---------------------------

def draw_skeleton(frame, kps):

    for x, y, c in kps:

        if c > 0.3:

            cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 0), -1)





# ---------------------------

# UI

# ---------------------------

def draw_ui(frame, state, robot_frame):



    h, w = frame.shape[:2]



    cv2.rectangle(frame, (10, 10), (450, 170), (0, 0, 0), -1)



    cv2.putText(frame, f"STATE: {state}", (20, 40),

                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)



    cv2.putText(frame, "UP = FORWARD / LEFT / RIGHT", (20, 70),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)



    cv2.putText(frame, "DOWN = BACKWARD", (20, 95),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)



    cv2.putText(frame, "WIDE = PICKUP", (20, 120),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)



    cv2.putText(frame, "TOGETHER = EXIT", (20, 145),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)



    col = lambda x: (0, 255, 0) if x else (80, 80, 80)



    cv2.circle(frame, (80, h - 60), 6, col(state == "FORWARD"), -1)

    cv2.circle(frame, (40, h - 40), 6, col(state == "LEFT"), -1)

    cv2.circle(frame, (120, h - 40), 6, col(state == "RIGHT"), -1)

    cv2.circle(frame, (80, h - 20), 6, col(state == "BACKWARD"), -1)



    if robot_frame is not None:

        rf = robot_frame

        rh, rw = rf.shape[:2]



        max_w = int(w * 0.5)

        max_h = int(h * 0.5)



        scale = min(max_w / rw, max_h / rh, 1.0)



        rf = cv2.resize(rf, (int(rw * scale), int(rh * scale)))



        rh, rw = rf.shape[:2]



        x1 = w - rw - 10

        y1 = 10



        frame[y1:y1+rh, x1:x1+rw] = rf





# ---------------------------

# MAIN LOOP

# ---------------------------

def run_pose_control(robot=None):



    model = YOLO("yolov8n-pose.pt")



    cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)



    history = []

    prev = "NONE"



    PICKUP_COOLDOWN = 3.0

    last_pickup = 0



    # FIXED STATE MACHINE

    pickup_state = 0

    pickup_time = 0



    print("🚀 SYSTEM STARTED")



    while True:

        ret, frame = cap.read()

        if not ret:

            break



        results = model.predict(frame, verbose=False, imgsz=320, conf=0.35)



        person = select_main_person(results)



        cmd = "NONE"



        if person is not None:

            kps = person

            kp_dict = {COCO_KEYPOINTS[i]: kps[i] for i in range(len(COCO_KEYPOINTS))}

            cmd = classify_pose(kp_dict)



            draw_skeleton(frame, kps)



        history.append(cmd)

        if len(history) > 6:

            history.pop(0)



        state = max(set(history), key=history.count)



        robot_frame = get_robot_frame(robot)



        draw_ui(frame, state, robot_frame)



        cv2.imshow("UGOT CONTROL", frame)



        # ---------------------------

        # ROBOT CONTROL

        # ---------------------------

        if robot is not None:

            try:

                now = time.time()



                if state == "FORWARD":

                    robot.mecanum_move_speed(0, 20)



                elif state == "BACKWARD":

                    robot.mecanum_move_speed(1, 20)



                elif state == "LEFT":

                    robot.mecanum_turn_speed(2, 30)



                elif state == "RIGHT":

                    robot.mecanum_turn_speed(3, 30)



                # ---------------------------

                # CLEAN PICKUP SEQUENCE FIXED

                # ---------------------------

                elif state == "PICKUP":

                    if pickup_state == 0 and now - last_pickup > PICKUP_COOLDOWN:

                        robot.mechanical_clamp_release()

                        robot.mechanical_joint_control(angle1=0, angle2=5, angle3=-75, duration=500)

                        time.sleep(1)



                        pickup_state = 1

                        pickup_time = now

                        last_pickup = now



                # STEP 2: WAIT 1 SEC THEN CLOSE

                if pickup_state == 1 and now - pickup_time > 1.0:

                    robot.mechanical_clamp_close()

                    pickup_state = 2

                    pickup_time = now



                # STEP 3: WAIT THEN RESET ARM

                if pickup_state == 2 and now - pickup_time > 1.0:

                    robot.mechanical_joint_control(angle1=0, angle2=60, angle3=-50, duration=1000)

                    pickup_state = 0



                if state not in ["FORWARD", "BACKWARD", "LEFT", "RIGHT", "PICKUP"]:

                    robot.mecanum_stop()



            except Exception as e:

                print("⚠️ Robot error:", e)



        if state == "EXIT":

            break



        if cv2.waitKey(1) & 0xFF == 27:

            break



    cap.release()

    cv2.destroyAllWindows()



    if robot is not None:

        robot.mecanum_stop()





if __name__ == "__main__":

    robot = init_robot("192.168.88.1")

    run_pose_control(robot)

Face

In [ ]:
import cv2

import numpy as np

import face_recognition

import time

import os

import sys

import warnings



warnings.filterwarnings("ignore", category=UserWarning)



try:

    from ugot import ugot

except ImportError:

    ugot = None



# ---------------------------

# SETTINGS

# ---------------------------

ROBOT_IP = "192.168.88.1"



try:

    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

except NameError:

    SCRIPT_DIR = os.getcwd()



# ---------------------------

# HELPER: GET ROBOT FRAME

# ---------------------------

def get_frame_from_robot(robot):

    frame = None

    try:

        if hasattr(robot, "get_video_frame"):

            frame = robot.get_video_frame()

        elif hasattr(robot, "get_camera_frame"):

            frame = robot.get_camera_frame()

        elif hasattr(robot, "read_camera_data"):

            raw = robot.read_camera_data()

            if raw is not None:

                nparr = np.frombuffer(raw, np.uint8)

                frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    except:

        pass

    return frame



# ---------------------------

# PRECISION DROP-OFF (FRONT OF PERSON)

# ---------------------------

def drop_apriltag(robot):

    print("\n[ACTION] Target Reached. Preparing drop...")

    robot.mecanum_stop()

    

    # 1. Back up first to ensure there is room for the arm to swing down

    print("[ACTION] Creating space in front of person...")

    robot.mecanum_move_speed_times(direction=1, speed=15, times=10, unit=1) 

    time.sleep(0.5)



    # 2. Lower the arm to ground level

    print("[ACTION] Lowering arm to ground...")

    robot.mechanical_joint_control(angle1=0, angle2=-35, angle3=-35, duration=1000)

    time.sleep(1.5)

    

    # 3. Release clamp

    print("[ACTION] Releasing AprilTag...")

    robot.mechanical_clamp_release()

    time.sleep(1)

    

    # 4. Back up to leave the tag standing alone

    print("[ACTION] Mission Complete. Moving away safely.")

    robot.mecanum_move_speed_times(direction=1, speed=20, times=12, unit=1)

    

    # 5. Reset Arm

    robot.mechanical_joint_control(angle1=0, angle2=40, angle3=40, duration=800)



# ---------------------------

# INIT ROBOT

# ---------------------------

def init_robot():

    if ugot is None: return None

    try:

        robot = ugot.UGOT()

        robot.initialize(ROBOT_IP)

        robot.load_models(["face_recognition", "word_recognition"])

        robot.open_camera()

        return robot

    except Exception as e:

        print(f"❌ Connection failed: {e}")

        return None



# ---------------------------

# LOAD DATABASE

# ---------------------------

def load_database():

    known_encodings = []

    known_names = []

    files = ["Arjun.jpg", "Ananya.jpg", "Coley.jpg", "Ryan.jpg"]

    for filename in files:

        img_path = os.path.join(SCRIPT_DIR, filename)

        if os.path.exists(img_path):

            img = face_recognition.load_image_file(img_path)

            encs = face_recognition.face_encodings(img)

            if encs:

                known_encodings.append(encs[0])

                known_names.append(filename.replace(".jpg", "").upper())

    return known_encodings, known_names



# ---------------------------

# MAIN PROCESS

# ---------------------------

def run_tracking():

    known_encs, known_names = load_database()

    if not known_encs:

        print("❌ No images found in script directory!")

        return



    print(f"\nTarget Database: {known_names}")

    target_input = input("Who should I find? ").strip().upper()

    

    robot = init_robot()

    if robot is None: return



    is_mission_complete = False

    has_found_target_once = False # STICKY LOCK FLAG

    

    print(f"🚀 SEARCHING FOR: {target_input}")



    while not is_mission_complete:

        frame = get_frame_from_robot(robot)

        if frame is None: continue



        # 1. OCR (Word Recognition)

        ocr_text = []

        try:

            res = robot.get_word_recognition_result()

            if res: 

                ocr_text = [str(w).upper() for w in res]

        except: pass



        # 2. FACE RECOGNITION (Full Res for Distance Detection > 15cm)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        locs = face_recognition.face_locations(rgb, model="hog")

        encs = face_recognition.face_encodings(rgb, locs)



        found_by_face = False

        found_by_text = any(target_input in word for word in ocr_text)

        target_box = None



        for (top, right, bottom, left), enc in zip(locs, encs):

            matches = face_recognition.compare_faces(known_encs, enc, tolerance=0.42)

            face_name = known_names[matches.index(True)] if True in matches else "UNKNOWN"

            

            if face_name == target_input:

                found_by_face = True

                target_box = (left, top, right, bottom)

            

            # Visuals

            color = (0, 255, 0) if (face_name == target_input) else (0, 0, 255)

            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)



        # 3. MOVEMENT & STICKY LOGIC

        if found_by_face or found_by_text:

            has_found_target_once = True # Target acquired!

            robot.mecanum_stop()

            

            if target_box:

                l, t, r, b = target_box

                cx = (l + r) // 2

                fw = r - l

            else:

                cx = frame.shape[1] // 2

                fw = 60 # Default for far away text match



            err = cx - (frame.shape[1] // 2)



            # Centering Phase

            if abs(err) > 35:

                # Use turn to center

                robot.mecanum_turn_speed(2 if err < 0 else 3, 12)

            # Approach Phase

            elif fw < 130: # STOP AT 130 (don't go too close or it loses detection)

                print(f"[MOVE] Sighted {target_input}. Approaching...")

                robot.mecanum_move_speed(0, 18)

            else:

                print(f"🎯 POSITIONED AT SAFE DISTANCE")

                drop_apriltag(robot)

                is_mission_complete = True

        

        else:

            # SEARCH MODE logic

            if not has_found_target_once:

                # If we haven't found them yet, keep spinning

                robot.mecanum_turn_speed(2, 10)

            else:

                # If we FOUND them but lost the frame, STOP and stay centered

                # Don't spin away! Just wait or strafe slightly to find them again.

                print(f"[LOCK] Lost {target_input} briefly. Waiting/Centering...")

                robot.mecanum_stop()



        # Debug Text

        if ocr_text:

            cv2.putText(frame, f"Text: {ocr_text}", (10, 30), 1, 1, (255, 255, 0), 2)



        cv2.imshow("Robot tracking", frame)

        if cv2.waitKey(1) & 0xFF == 27: break



    robot.mecanum_stop()

    cv2.destroyAllWindows()



if __name__ == "__main__":

    run_tracking()

TEST